[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week10.ipynb)

# Week 10: Recurrent Networks (RNNs)
**Introduction to Deep Learning (HIT)** &middot; Part III: Architectures & Representation Learning

Sequence data and recurrence; the RNN cell; backpropagation through time; vanishing and exploding gradients.

**Instructor practice notebook** for the 2-hour practice lesson. Work through the sections below live, running each cell and trying the variations. The student homework is the weekly lab.

### Goals

- Build a plain RNN for sequence data.
- Understand recurrence and backpropagation through time.
- Observe the vanishing-gradient problem directly.

### Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## 1. An RNN over a sequence
It carries a hidden state forward, reusing the same weights at every step.

In [ ]:
torch.manual_seed(0)
rnn = nn.RNN(input_size=1, hidden_size=16, batch_first=True)
x = torch.randn(8, 20, 1)                 # (batch, time, features)
out, h = rnn(x)
print("output sequence:", tuple(out.shape), "| final hidden:", tuple(h.shape))
print("the same weight is used at all 20 steps:", tuple(rnn.weight_hh_l0.shape))

## 2. Vanishing gradients, measured
Gradient of the LAST output w.r.t. each input step shrinks for early steps.

In [ ]:
T = 40
rnn = nn.RNN(1, 8, batch_first=True)
x = torch.randn(1, T, 1, requires_grad=True)
out, _ = rnn(x)
out[:, -1].sum().backward()
g = x.grad.abs().squeeze().tolist()
plt.plot(range(T), g); plt.xlabel("time step"); plt.ylabel("|grad of last output|")
plt.title("Early steps receive tiny gradients"); plt.show()
print("grad at step 0:", f"{g[0]:.2e}", "| grad at last step:", f"{g[-1]:.2e}")

## 3. Gradient clipping
Cap the global gradient norm so a spike cannot blow up training.

In [ ]:
rnn = nn.RNN(1, 8, batch_first=True); o = torch.optim.SGD(rnn.parameters(), 0.1)
x = torch.randn(4, 30, 1); y = torch.randn(4, 30, 8)
o.zero_grad(); out, _ = rnn(x); ((out - y) ** 2).mean().backward()
norm = torch.nn.utils.clip_grad_norm_(rnn.parameters(), max_norm=1.0)
print("global grad norm before clip:", round(norm.item(), 3), "-> clipped to <= 1.0")

## 4. A tiny sequence task end to end
Predict whether a short random walk ends positive.

In [ ]:
torch.manual_seed(0)
seqs = torch.randn(400, 15, 1)
labels = (seqs.sum(dim=1).squeeze() > 0).long()
class SeqClf(nn.Module):
    def __init__(self):
        super().__init__(); self.rnn = nn.RNN(1, 16, batch_first=True); self.head = nn.Linear(16, 2)
    def forward(self, x):
        out, _ = self.rnn(x); return self.head(out[:, -1])     # last hidden -> 2 classes
m = SeqClf(); o = torch.optim.Adam(m.parameters(), 0.01); f = nn.CrossEntropyLoss()
for _ in range(120):
    o.zero_grad(); f(m(seqs), labels).backward(); o.step()
print("accuracy:", round((m(seqs).argmax(1) == labels).float().mean().item(), 3))

**Try it live:** raise the sequence length to 60 and see the accuracy drop, the vanishing-gradient problem in action.

---
Student materials for this week: the lab handout (`labs/week10.html`) and the curated references (`references/week10.html`) in the course site.